In [ ]:
from google.colab import drive
import glob, os
drive.mount('/content/drive')
NAME = '*'
pats = ['/content/drive/MyDrive/' + '*/' * d + NAME + '/run_all.py' for d in range(0, 3)]
pats += ['/content/drive/Othercomputers/' + '*/' * d + NAME + '/run_all.py' for d in range(1, 5)]
cands = sorted({p for pat in pats for p in glob.glob(pat)}, key=os.path.getmtime)
if not cands:
    raise FileNotFoundError('找不到 run_all.py，請確認專案資料夾已同步完成')
exec(open(cands[-1], encoding='utf-8').read())


In [ ]:
import glob, os, re, shutil, subprocess, sys, time, urllib.request
from google.colab import drive

drive.mount('/content/drive')

def sh(cmd, cwd=None):
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()

PROJECT_NAME = '*'
patterns = ['/content/drive/MyDrive/' + '*/' * d + PROJECT_NAME + '/setup_colab.sh' for d in range(0, 3)]
patterns += ['/content/drive/Othercomputers/' + '*/' * d + PROJECT_NAME + '/setup_colab.sh' for d in range(1, 5)]
found = sorted({os.path.dirname(p) for pat in patterns for p in glob.glob(pat)})
if not found:
    raise FileNotFoundError('在「我的雲端硬碟」和「電腦」裡都找不到專案資料夾（裡面要有 setup_colab.sh），請確認已同步完成')

def newest_code_mtime(d):
    times = [os.path.getmtime(p) for sub in ('backend/app', 'frontend', 'scripts')
             for p in glob.glob(os.path.join(d, sub, '**', '*.*'), recursive=True)]
    return max(times, default=os.path.getmtime(os.path.join(d, 'setup_colab.sh')))

found.sort(key=newest_code_mtime)
SOURCE_DIR = found[-1]
if len(found) > 1:
    print('⚠️ 找到不只一份專案，使用程式碼最新的那一份（建議把其他舊的刪掉，避免混淆）：')
    for f in found:
        print('   ', '👉' if f == SOURCE_DIR else '  ', f, time.strftime('%m/%d %H:%M', time.localtime(newest_code_mtime(f))))

PROJECT_DIR = '/content/meetsum'
BACKEND_DIR = os.path.join(PROJECT_DIR, 'backend')
LOG_DIR = os.path.join(PROJECT_DIR, 'logs')

def sync_code():
    shutil.copytree(SOURCE_DIR, PROJECT_DIR, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('logs', 'audio', '__pycache__', '*.pyc', '.env'))
    env_dst = os.path.join(PROJECT_DIR, '.env')
    for name in ('.env', '.env.example'):
        env_src = os.path.join(SOURCE_DIR, name)
        if os.path.exists(env_src):
            shutil.copy2(env_src, env_dst)
            break
    os.makedirs(LOG_DIR, exist_ok=True)
    os.makedirs(os.path.join(BACKEND_DIR, 'storage', 'audio'), exist_ok=True)

sync_code()
os.chdir(PROJECT_DIR)
print('✅ 專案來源：', SOURCE_DIR)
print('✅ 執行位置：', PROJECT_DIR)


In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

SETUP = os.path.join(PROJECT_DIR, 'setup_colab.sh')

with open(SETUP, 'rb') as f:
    data = f.read()
with open(SETUP, 'wb') as f:
    f.write(data.replace(b'\r\n', b'\n'))

code = sh(['bash', SETUP])
print('\n✅ 安裝完成' if code == 0 else f'\n❌ 安裝失敗（代碼 {code}），請把上面最後幾十行貼出來')


In [ ]:
import json, os, re, shutil, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

sync_code()

for name in ('.env', '.env.example'):
    if os.path.exists(os.path.join(SOURCE_DIR, name)):
        shutil.copy2(os.path.join(SOURCE_DIR, name), os.path.join(PROJECT_DIR, '.env'))
        break
print('已複製最新程式：', SOURCE_DIR, '→', PROJECT_DIR)

subprocess.run(['pkill', '-f', 'uvicorn app[.]main'])
time.sleep(1)

log = open(os.path.join(LOG_DIR, 'backend.log'), 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND_DIR, stdout=log, stderr=subprocess.STDOUT,
)

print('等待後端啟動', end='', flush=True)
ready = False
for _ in range(60):
    if server.poll() is not None:
        break
    try:
        health = json.load(urllib.request.urlopen('http://127.0.0.1:8000/api/v1/health', timeout=2))
        ready = True
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
print()

if ready:
    print('✅ 後端已啟動（帳號 demo / 密碼 demo1234）')
    print(f"   LLM：{health.get('llm_model')}　溫度 {health.get('llm_temperature')}　seed {health.get('llm_seed')}"
          f"　思考 {health.get('llm_think')}　上下文 {health.get('llm_num_ctx')}")
    print('   ※ 如果下面什麼都沒出現（例如用 VS Code），請改執行第 5 段取得網址')
    try:
        import warnings
        from google.colab import output
        output.serve_kernel_port_as_iframe(8000, height=900)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            output.serve_kernel_port_as_window(8000)
    except Exception as e:
        print('   （無法在這裡顯示網頁：', e, '）')
else:
    print('❌ 後端沒有啟動成功，以下是 log 最後 30 行：')
    with open(os.path.join(LOG_DIR, 'backend.log'), encoding='utf-8', errors='replace') as f:
        print(''.join(f.readlines()[-30:]))


In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

with open(os.path.join(LOG_DIR, 'backend.log'), encoding='utf-8', errors='replace') as f:
    print(''.join(f.readlines()[-40:]))


In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

CLOUDFLARED = '/usr/local/bin/cloudflared'
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)

subprocess.run(['pkill', '-f', 'cloudflared[ ]tunnel'])
tunnel_log_path = '/content/tunnel.log'
tunnel = subprocess.Popen([CLOUDFLARED, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=open(tunnel_log_path, 'w'), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    text = open(tunnel_log_path).read()
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
    url = m.group(0) if m else None
    if url and 'Registered tunnel connection' in text:
        break

ok = False
if url:
    print('網址已建立，確認外部連得到', end='', flush=True)
    for _ in range(45):
        try:
            urllib.request.urlopen(url + '/api/v1/health', timeout=5)
            ok = True
            break
        except Exception:
            print('.', end='', flush=True)
            time.sleep(2)
    print()
if ok:
    print('👉 測試用網址：', url)
    print('   若顯示「無法連上這個網站」：等 30 秒再重新整理；仍不行，在 Windows 命令提示字元執行 ipconfig /flushdns')
    print('   每次重跑這段都會換新網址，舊網址會失效')
else:
    print('❌ 臨時網址沒有建立成功，log 如下：')
    print(open(tunnel_log_path).read()[-2000:])


In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

r1 = subprocess.run(['pkill', '-f', 'uvicorn app[.]main'])
print('後端已關閉' if r1.returncode == 0 else '後端沒有在執行')
r2 = subprocess.run(['pkill', '-f', 'cloudflared[ ]tunnel'])
print('臨時網址已關閉' if r2.returncode == 0 else '臨時網址沒有在執行')


In [ ]:
import os, re, subprocess, sys, time, urllib.request
if 'SOURCE_DIR' not in globals():
    raise RuntimeError('請先執行第 1 段（掛載 Drive、找到專案資料夾），再執行這一段')

DB_CMD = ['psql', '-h', 'localhost', '-U', 'meeting', '-d', 'meeting_assistant', '-c']
env = dict(os.environ, PGPASSWORD='meeting')
for sql in [
    "SELECT meeting_name, source_type, status, created_at FROM meetings ORDER BY created_at DESC;",
    "SELECT m.meeting_name, t.transcript->>'segment_count' AS 段數 FROM meeting_transcripts t JOIN meetings m ON m.id = t.meeting_id;",
]:
    print(subprocess.run(DB_CMD + [sql], env=env, capture_output=True, text=True).stdout)
